# Stage 1 - Phase 7: V2 V-JEPA 2.1-B (video branch)Run this **after** the VideoMAEv2-B pipeline in notebook 01 is validated: thedata path, split and evaluation are identical, so only the backbone changes.Checkpoint facts (verified against the official repository, not assumed):* V-JEPA 2.1 (2026-03-16) is the only V-JEPA 2.x generation with a **Base**  encoder; the ViT-B/16 checkpoint is `vjepa2_1_vitb_dist_vitG_384.pt`  (distilled from ViT-G, `checkpoint_key="ema_encoder"`);* `transformers` cannot load V-JEPA 2.1, so the encoder is built from a local  clone of `facebookresearch/vjepa2` (`app/vjepa_2_1/models/vision_transformer.py`)  with the exact encoder kwargs of `src/hub/backbones.py`;* the checkpoint is loaded from a **local file** with `strict=True`. Downloading  is opt-in, because the submission environment may be offline - and the  upstream `torch.hub` path is currently broken (`VJEPA_BASE_URL` is left at  `http://localhost:8300` in the released `backbones.py`);* high spatial resolution is this backbone's advantage, so the default input  geometry is 384. RoPE with `interpolate_rope=True` makes the frame count  flexible, so `input_frames` may be lower than the 64 the checkpoint was built  with. Both stay configurable.Set `model.params.source_root` and `model.params.checkpoint_path` in`configs/stage1/vjepa2_1_b.yaml` before running.

## 1. Setup

In [ ]:
from __future__ import annotationsimport sysfrom pathlib import Pathimport numpy as npimport pandas as pdimport torchimport yaml# The package is expected to be installed with `pip install -e .` from the# repository root. The fallback keeps a fresh clone usable without installing.try:    import blackbox_detection  # noqa: F401except ModuleNotFoundError:    _root = Path.cwd()    while _root != _root.parent and not (_root / "pyproject.toml").is_file():        _root = _root.parent    sys.path.insert(0, str(_root / "src"))from blackbox_detection.utils import seed_everything, setup_loggerprint("torch", torch.__version__, "| cuda", torch.cuda.is_available())

In [ ]:
import jsonfrom blackbox_detection.stage1 import (    AggregationConfig,    ClipAugmentConfig,    Stage1Evaluator,    Stage1Trainer,    Stage1VideoDataset,    TrainConfig,    ValidationSubsetSpec,    apply_split,    build_clip_sampler,    build_dataloader,    build_stage1_model,    build_validation_subsets,    build_video_transforms,    load_manifest,    load_split,    save_predictions,    search_best_threshold,    video_batch_adapter,)from blackbox_detection.stage1.evaluator import probabilities_to_labelsfrom blackbox_detection.utils import load_checkpoint, stage1_scorelogger = setup_logger("stage1.vjepa2_1_b")

## 2. Paths

In [ ]:
REPO_ROOT = Path.cwd()while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "pyproject.toml").is_file():    REPO_ROOT = REPO_ROOT.parentCONFIG_DIR = REPO_ROOT / "configs" / "stage1"OUTPUT_ROOT = REPO_ROOT / "outputs" / "stage1"OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)print("repo   :", REPO_ROOT)print("configs:", CONFIG_DIR)print("outputs:", OUTPUT_ROOT)

## 3. Config

In [ ]:
CONFIG = yaml.safe_load((CONFIG_DIR / "vjepa2_1_b.yaml").read_text(encoding="utf-8"))MODEL_NAME = CONFIG["model"]["name"]ADAPTER = video_batch_adapter()params = CONFIG["model"]["params"]if params.get("source_root") is None:    raise ValueError(        "Set model.params.source_root in configs/stage1/vjepa2_1_b.yaml to a local "        "clone of https://github.com/facebookresearch/vjepa2 - V-JEPA 2.1 is not "        "supported by transformers."    )if params.get("checkpoint_path") is None and not params.get("allow_download", False):    raise ValueError(        "Set model.params.checkpoint_path to a local copy of "        "vjepa2_1_vitb_dist_vitG_384.pt, or set allow_download: true."    )SEED = int(CONFIG["train"]["seed"])RUN_DIR = REPO_ROOT / CONFIG["train"]["output_dir"]RUN_DIR.mkdir(parents=True, exist_ok=True)seed_everything(SEED, deterministic=False)print(MODEL_NAME, "->", RUN_DIR)print(json.dumps(params, indent=2))

## 4. Data### 4.1 Fixed manifest and split

In [ ]:
DATA_CONFIG = yaml.safe_load((CONFIG_DIR / "dlc2021.yaml").read_text(encoding="utf-8"))MANIFEST_PATH = REPO_ROOT / DATA_CONFIG["paths"]["manifest"]SPLIT_PATH = REPO_ROOT / DATA_CONFIG["paths"]["split"]manifest = load_manifest(MANIFEST_PATH, check_paths_exist=False)split = load_split(SPLIT_PATH)splits = apply_split(manifest, split)train_manifest, val_manifest = splits["train"], splits["val"]assert not set(train_manifest["video_id"]) & set(val_manifest["video_id"])print("train videos:", len(train_manifest), "| val videos:", len(val_manifest))print(train_manifest["label"].value_counts().to_dict(), val_manifest["label"].value_counts().to_dict())

In [ ]:
subset_specs = [    ValidationSubsetSpec(        name=spec["name"],        column=spec["column"],        minimum=spec.get("minimum"),        maximum=spec.get("maximum"),        min_per_class=int(spec.get("min_per_class", 20)),    )    for spec in DATA_CONFIG.get("validation_subsets", [])]# VAL-A is the whole stratified validation set; the specs above are VAL-B style# controlled diagnostics, reported only when they hold enough videos per class.VAL_SUBSETS = {"val_a": val_manifest["video_id"].tolist()}for name, result in build_validation_subsets(val_manifest, subset_specs).items():    if result.usable:        VAL_SUBSETS[name] = result.video_ids    else:        print(f"skipping {name}: {result.reason}")print("evaluation subsets:", {name: len(ids) for name, ids in VAL_SUBSETS.items()})

## 5. Model

In [ ]:
model = build_stage1_model(    MODEL_NAME,    finetune_mode=CONFIG["model"]["finetune_mode"],    unfreeze_last_n=int(CONFIG["model"]["unfreeze_last_n"]),    **params,)from blackbox_detection.stage1.models import count_parametersprint("blocks:", len(model.blocks), "| feature dim:", model.feature_dim)print("parameters:", count_parameters(model))print("load report:", json.dumps(model.load_report["weights"], indent=2, default=str))# The load must have come from the official checkpoint, not from random weights.assert "random init" not in str(model.load_report["weights"]["source"])

### 5.1 Datasets and loaders384-resolution clips are heavy; keep `batch_size` small and use gradient accumulation.

In [ ]:
video_config = CONFIG["data"]augmentation_config = CONFIG["augmentation"]preprocessing = model.preprocessing()print("checkpoint preprocessing:", preprocessing)# Normalisation and geometry come from the checkpoint, never from a guess.train_transform, val_transform = build_video_transforms(    crop_size=int(preprocessing["input_size"]),    mean=tuple(preprocessing["mean"]),    std=tuple(preprocessing["std"]),    train_config=ClipAugmentConfig(        crop_size=int(preprocessing["input_size"]),        scale_range=tuple(augmentation_config["scale_range"]),        ratio_range=tuple(augmentation_config["ratio_range"]),        hflip_prob=float(augmentation_config["hflip_prob"]),        brightness=float(augmentation_config["brightness"]),        contrast=float(augmentation_config["contrast"]),        perspective_prob=float(augmentation_config["perspective_prob"]),        perspective_scale=float(augmentation_config["perspective_scale"]),    ),)train_dataset = Stage1VideoDataset(    train_manifest,    clip_sampler=build_clip_sampler(        train=True,        num_frames=int(video_config["num_frames"]),        strides=video_config["train_strides"],        num_clips=int(video_config["train_num_clips"]),    ),    transform=train_transform,    on_error="zero",    deterministic=False,)val_dataset = Stage1VideoDataset(    val_manifest,    clip_sampler=build_clip_sampler(        train=False,        num_frames=int(video_config["num_frames"]),        val_stride=int(video_config["val_stride"]),        num_clips=int(video_config["val_num_clips"]),    ),    transform=val_transform,    on_error="zero",    deterministic=True,)train_loader = build_dataloader(    train_dataset,    batch_size=int(video_config["batch_size"]),    shuffle=True,    num_workers=int(video_config["num_workers"]),    seed=SEED,    drop_last=True,)val_loader = build_dataloader(    val_dataset,    batch_size=int(video_config["val_batch_size"]),    shuffle=False,    num_workers=int(video_config["num_workers"]),    seed=SEED,)batch = next(iter(train_loader))print("clip batch:", tuple(batch["pixels"].shape), "| labels:", batch["label"].tolist())

In [ ]:
# Clip-consistent augmentation check: a static clip must stay static after# augmentation. Per-frame jitter would fabricate the very flicker the video# branch is meant to detect in real re-recordings.clip = batch["pixels"][0, 0]static = np.repeat(np.full((1, 240, 320, 3), 128, dtype=np.uint8), 8, axis=0)augmented = train_transform(static, np.random.default_rng(0))spread = float(augmented.mean(dim=(0, 2, 3)).std())print("per-frame mean spread on a static clip:", spread)assert spread < 1e-5, "augmentation is not clip-consistent"

## 6. Training

In [ ]:
train_config = CONFIG["train"]trainer_config = TrainConfig(    epochs=int(train_config["epochs"]),    learning_rate=float(train_config["learning_rate"]),    head_learning_rate=(        float(train_config["head_learning_rate"])        if train_config.get("head_learning_rate") is not None        else None    ),    weight_decay=float(train_config["weight_decay"]),    warmup_ratio=float(train_config["warmup_ratio"]),    grad_accum_steps=int(train_config["grad_accum_steps"]),    max_grad_norm=float(train_config["max_grad_norm"]),    amp=bool(train_config["amp"]),    label_smoothing=float(train_config.get("label_smoothing", 0.0)),    early_stopping_patience=int(train_config["early_stopping_patience"]),    eval_every=int(train_config["eval_every"]),    seed=SEED,    output_dir=RUN_DIR,    model_name=MODEL_NAME,    wandb_enabled=False,)trainer = Stage1Trainer(    model,    trainer_config,    adapter=ADAPTER,    aggregation=AggregationConfig(        frame_method=CONFIG["evaluation"]["aggregation"]["frame_method"],        video_method=CONFIG["evaluation"]["aggregation"]["video_method"],    ),    model_config={"name": MODEL_NAME, "params": CONFIG["model"]["params"]},)print("device:", trainer.device, "| amp:", trainer.amp)outcome = trainer.fit(train_loader, val_loader, subsets=VAL_SUBSETS)print()print(f"best epoch {outcome.best_epoch}: Macro-F1 {outcome.best_macro_f1:.4f} "      f"at threshold {outcome.best_threshold:.3f}")

## 7. Validation

In [ ]:
# Re-evaluate the best checkpoint explicitly, so the reported numbers come from# the weights that were actually saved.load_checkpoint(RUN_DIR / "best.pt", model=model, map_location=trainer.device,                restore_rng_state=False)evaluator = Stage1Evaluator(    model,    ADAPTER,    device=trainer.device,    amp=trainer.amp,    aggregation=trainer.aggregation,)result, units = evaluator.evaluate(val_loader, subsets=VAL_SUBSETS, return_units=True)print(f"Macro-F1            : {result.macro_f1:.4f}  (official Stage 1 metric)")print(f"Macro-F1 @ thr 0.5  : {result.macro_f1_at_default:.4f}")print(f"optimal threshold   : {result.threshold:.4f}")print(f"class-wise F1       : {result.per_class_f1}")print(f"per-dataset Macro-F1: {result.dataset_scores}")print(f"videos              : {result.num_videos} ({result.num_invalid_videos} with decode problems)")print()for name, payload in result.subset_scores.items():    print(f"{name}: {payload}")

### 7.1 Threshold search

In [ ]:
# Threshold sweep: 0.5 is only a starting point. A threshold far from 0.5, or a# sharp peak, is itself a signal about calibration and class balance.labels = result.predictions["label"].tolist()probabilities = result.predictions["prob_rerecorded"].to_numpy()sweep = pd.DataFrame(    {        "threshold": np.round(np.arange(0.05, 1.0, 0.05), 2),    })sweep["macro_f1"] = [    stage1_score(labels, probabilities_to_labels(probabilities, threshold))    for threshold in sweep["threshold"]]display(sweep.set_index("threshold").T)best_threshold, best_score = search_best_threshold(labels, probabilities)print(f"searched threshold {best_threshold:.4f} -> Macro-F1 {best_score:.4f}")

## 8. Results

In [ ]:
display(outcome.history)print("Validation predictions (video level):")display(result.predictions.head(10))print("Per-unit score spread per video (aggregation sanity):")spread = (    units.groupby("video_id")["prob_rerecorded"]    .agg(["count", "mean", "std", "min", "max"])    .head(10))display(spread)

### 8.1 How to read these numbers

**Do not pick the final model on this number alone.** A very high DLC-2021Macro-F1 can mean the model learned a document / display / resolution shortcutrather than a recapture representation. What to record for each run:* Macro-F1 and class-wise F1,* the optimal threshold and how sharp the sweep peak is,* overfitting behaviour across epochs (`history.csv`),* VAL-A versus the controlled VAL-B subset,* prediction diversity against the other models (notebook 04).`val_predictions.csv` is the artefact that makes all of this comparable later,and `best.pt` is reusable as a DLC-pretrained initialisation once the paired CCDre-recordings exist.

## 9. Save

In [ ]:
save_predictions(result.predictions, RUN_DIR / "val_predictions.csv")outcome.history.to_csv(RUN_DIR / "history.csv", index=False)summary = {    "model_name": MODEL_NAME,    "val_macro_f1": float(result.macro_f1),    "val_macro_f1_at_0.5": float(result.macro_f1_at_default),    "best_threshold": float(result.threshold),    "per_class_f1": result.per_class_f1,    "best_epoch": int(outcome.best_epoch),    "num_val_videos": int(result.num_videos),    "subset_scores": result.subset_scores,    "preprocessing": dict(model.preprocessing()),}(RUN_DIR / "summary.json").write_text(json.dumps(summary, indent=2, default=str), encoding="utf-8")print("artefacts in", RUN_DIR)for path in sorted(RUN_DIR.iterdir()):    print("  ", path.name)

In [ ]:
print("Next: scripts/stage1/04_compare_and_fuse.ipynb")